# ODM Assignment 2

After coding the ODM for the first assignment 1, this assignment implements the usage of it for MongoDB queries.

## TODO

- Query 1: Grouping only shows name field
- Query 2:
- Query 3:
- Query 4:
- Query 5:
- Query 6:
- Query 7:


## Inserting data

First, we initiate the ODM with `initApp` and load in the data from it. The database data is extracted into JSON files that are then used to insert into a new database if this one doesn't have it.

In [1]:
import json
import ODM
import time

from ODM import initApp
from tqdm.notebook import tqdm

# Globals
import_data = False
scope = {}

# Initialize DB
initApp(definitions_path="data/models.yml", db_name="abd", scope=scope)

# Get model classes
User = scope.get("User")
Company = scope.get("Company")
Educational_Center = scope.get("Educational_Center")

# Load JSON data
with open("data/users.json", "r", encoding="utf-8") as f:
    users_data = json.load(f)
with open("data/companies.json", "r", encoding="utf-8") as f:
    companies_data = json.load(f)
with open("data/educational_centers.json", "r", encoding="utf-8") as f:
    ed_centers_data = json.load(f)

# Helper to filter admissible/required vars
def valid_args(model, d):
    valid = model._required_vars.union(model._admissible_vars)
    return {k: v for k, v in d.items() if k in valid}

# Introduce data if user sets introduce data variable to true
print("\n=== Importing data into MongoDB ===")
if import_data:
    users = []
    for u in tqdm(users_data, desc = "Adding users"):
        args = valid_args(User, u)
        user = User(**args)
        user.save()
        users.append(user)
        time.sleep(0.01)

    # Test Company
    companies = []
    for c in tqdm(companies_data, desc = "Adding companies"):
        args = valid_args(Company, c)
        company = Company(**args)
        company.save()
        companies.append(company)
        time.sleep(0.01)

    # Test Educational_Center
  
    ed_centers = []
    for ec in tqdm(ed_centers_data, desc = "Adding educational centers"):
        args = valid_args(Educational_Center, ec)
        ed_center = Educational_Center(**args)
        ed_center.save()
        ed_centers.append(ed_center)
        time.sleep(0.01)

Connected to database: abd
Successfully connected to MongoDB!
Loading schema from data/models.yml
Initializing model: User
Initializing model: Company
Initializing model: Educational_Center

=== Importing data into MongoDB ===


## Query 1

**List of all people who have studied at the UPM or UAM.**

- Join the users and the ed_center on the educational_centers.email.
- Separate the ed_details into documents.
- Select documents where ed_center name is UPM or UAM.
- Group by user name to avoid duplication.

In [14]:
query1 = User.aggregate([
  {
    "$lookup": {
      "from": "Educational_Center",
      "localField": "educational_centers.email",
      "foreignField": "email",
      "as": "ed_details",
    },
  },
  { "$unwind": "$ed_details" },
  {
    "$match": {
      "ed_details.name": {
        "$in": [
          "Universidad Politécnica de Madrid",
          "Universidad Autónoma de Madrid",
        ],
      },
    },
  },
  { "$group": { "_id": "$_id", "user": { "$first": "$$ROOT"  }}},
  { "$replaceRoot": { "newRoot": "$user" }},
  {
    "$project": {
      "ed_details": 0,
    }
  }
]);

print(f"Query 1: List of all people who have studied at the UPM or UAM")
for d in query1:
    print(d)

Query 1: List of all people who have studied at the UPM or UAM
{'_id': ObjectId('690388331e8847783aed083a'), 'name': 'María Fernández', 'email': 'user1@ucm.es', 'address': 'Calle Gran Vía, 1, Madrid, España', 'interests': 'reading,music,hiking', 'skills': 'Java,Python,project management', 'educational_centers': [{'email': 'info@ucm.es', 'completion_date': '2018'}, {'email': 'info@upm.es', 'completion_date': '2021'}], 'companies': ['contacto@tecnologiasavanzadas.com'], 'description': 'Graduate in Computer Science from UCM, works in project management and is interested in Artificial Intelligence and data-driven projects.', 'address_loc': {'type': 'Point', 'coordinates': [-3.872021, 40.473748]}}
{'_id': ObjectId('690397928b5904b9d03a3c0f'), 'name': 'Juan Castillo', 'email': 'user26@ucm.es', 'address': 'Calle Gran Vía, 26, Madrid, España', 'interests': 'reading,music,hiking', 'skills': 'Java,Python,project management', 'educational_centers': [{'email': 'info@ucm.es', 'completion_date': '20

# Query 2

**Different universities where people residing in Madrid have studied.**

- Filter users who live in Madrid.
- Join with Educational_Center using each center's email.
- Unwind joined ed_details into separate docs.
- Group by educational center name and return distinct names.

In [15]:
query2 = Educational_Center.aggregate([
  {
    "$lookup": {
      "from": "User",
      "localField": "email",                
      "foreignField": "educational_centers.email", 
      "as": "users",
    },
  },
  { "$unwind": "$users" },
  {
    "$match": {
      "users.address": { "$regex": "Madrid", "$options": "i" }, 
    },
  },
  {
    "$group": {
      "_id": "$_id",                       
      "educational_center": { "$first": "$$ROOT" }   
    },
  },
  { "$replaceRoot": { "newRoot": "$educational_center" }},
  {
    "$project": {
        "users": 0,
    }
  }
]);

print(f"Query 2: Different universities where people residing in Madrid have studied")
for d in query2:
    print(d)

Query 2: Different universities where people residing in Madrid have studied
{'_id': ObjectId('6903983d8b5904b9d03a3c84'), 'name': 'Universidad Politécnica de Madrid', 'address': 'Ciudad Universitaria (Av. Ramiro de Maeztu), 7, 28040 Madrid, España', 'email': 'info@upm.es', 'address_loc': {'type': 'Point', 'coordinates': [0, 0]}}
{'_id': ObjectId('690398398b5904b9d03a3c70'), 'name': 'Universidad Complutense de Madrid', 'address': 'Madrid, España', 'email': 'info@ucm.es', 'address_loc': {'type': 'Point', 'coordinates': [-3.703507, 40.416782]}}
{'_id': ObjectId('690398398b5904b9d03a3c74'), 'name': 'Instituto de Formación Professional López', 'address': 'Madrid, España', 'email': 'info@ifpl.es', 'address_loc': {'type': 'Point', 'coordinates': [-3.703507, 40.416782]}}
{'_id': ObjectId('690398418b5904b9d03a3c85'), 'name': 'Universidad Autónoma de Madrid', 'address': 'Campus de Cantoblanco, 28049 Madrid, España', 'email': 'info@uam.es', 'address_loc': {'type': 'Point', 'coordinates': [0, 0]}

# Query 3

**People whose profile description includes "Big Data" or
"Artificial Intelligence".**

- Use `$match` with `$or` to find descriptions containing the terms.
- Use case-insensitive regex to catch variations of casing.

In [16]:
query3 = User.aggregate([
  {
    "$match": {
      "$or": [
        { "description": { "$regex": "Big Data", "$options": "i" } },
        { "description": { "$regex": "Artificial Intelligence", "$options": "i" } },
      ],
    },
  }, { "$group": { "_id": "$_id", "user": { "$first": "$$ROOT" } }},
  { "$replaceRoot": { "newRoot": "$user" }},
]);

print(f"Query 3: People whose profile description includes 'Big Data' or 'Artificial Intelligence'")
for d in query3:
    print(d)

Query 3: People whose profile description includes 'Big Data' or 'Artificial Intelligence'
{'_id': ObjectId('690397c08b5904b9d03a3c28'), 'name': 'Sofía Castillo', 'email': 'user51@ucm.es', 'address': 'Calle Gran Vía, 51, Madrid, España', 'interests': 'reading,music,hiking', 'skills': 'Java,Python,project management', 'educational_centers': [{'email': 'info@ucm.es', 'completion_date': '2023'}, {'email': 'info@upm.es', 'completion_date': '2025'}], 'companies': ['contacto@tecnologiasavanzadas.com', 'info@google.es'], 'description': "Master's student studying Human-Computer Interaction; interested in Artificial Intelligence applied to UX.", 'address_loc': {'type': 'Point', 'coordinates': [-3.871477, 40.471293]}}
{'_id': ObjectId('6903977e8b5904b9d03a3c02'), 'name': 'Patricia Ruiz', 'email': 'user13@itgranada.es', 'address': 'Calle Gran Capitán, 13, Granada, España', 'interests': 'technology,reading,cooking', 'skills': 'Python,JavaScript,SQL', 'educational_centers': [{'email': 'info@itgrana

# Query 4

**Save users who completed any study in 2017 or later into a new
collection.**

- Match users where at least one educational_centers entry has a
  completion_date $\geq$ 2017.
- Output results to a new collection using `$out`.

Note: If completion_date is stored as a string year, the first
pipeline works. If stored as a date, use the ISODate variant.

String/year version:

In [9]:
# Collection for output
col2017 = "users_completed_studies_in_2017_or_after"

User.aggregate([
  {
    "$match": {
      "educational_centers.completion_date": { "$gte": "2017" },
    },
  },
  { "$out": col2017 },
]);

# Run it again to print out results
query4 = User.aggregate([
  {
    "$match": {
      "educational_centers.completion_date": { "$gte": "2017" },
    },
  },
  { "$project": { "name": 1 }},
]);

for d in query4:
    print(d)

{'_id': ObjectId('690388331e8847783aed083a'), 'name': 'María Fernández'}
{'_id': ObjectId('690388341e8847783aed083b'), 'name': 'Carlos García'}
{'_id': ObjectId('690388361e8847783aed083c'), 'name': 'Lucía Martínez'}
{'_id': ObjectId('69038e31f064b0915f86ed23'), 'name': 'Javier López'}
{'_id': ObjectId('690397748b5904b9d03a3bfb'), 'name': 'Pedro Sánchez'}
{'_id': ObjectId('690397768b5904b9d03a3bfc'), 'name': 'Elena Torres'}
{'_id': ObjectId('6903977a8b5904b9d03a3bff'), 'name': 'Diego Navarro'}
{'_id': ObjectId('6903977b8b5904b9d03a3c00'), 'name': 'Laura González'}
{'_id': ObjectId('6903977e8b5904b9d03a3c02'), 'name': 'Patricia Ruiz'}
{'_id': ObjectId('690397808b5904b9d03a3c03'), 'name': 'David Sánchez'}
{'_id': ObjectId('690397818b5904b9d03a3c04'), 'name': 'Cristina Herrera'}
{'_id': ObjectId('690397898b5904b9d03a3c0a'), 'name': 'Beatriz Pérez'}
{'_id': ObjectId('6903978e8b5904b9d03a3c0c'), 'name': 'Rocío Herrera'}
{'_id': ObjectId('690397918b5904b9d03a3c0e'), 'name': 'Marta Mendoza'}
{

# Query 5

**Average number of studies for people who have worked at Microsoft.**

- Join users with Company collection via companies (company emails).
- Unwind company details and match company name to Microsoft.
- Group and compute the average number of educational_centers per user.

In [6]:
query5 = User.aggregate([
  {
    "$lookup": {
      "from": "Company",
      "localField": "companies",
      "foreignField": "email",
      "as": "c_details",
    },
  },
  { "$unwind": "$c_details" },
  { "$match": { "c_details.name": { "$regex": "Microsoft", "$options": "i" } } },
  {
    "$group": {
      "_id": "microsoft_workers",
      "average_studies": { "$avg": { "$size": "$educational_centers" } },
    },
  },
]);

for d in query5:
    print(d)

{'_id': 'microsoft_workers', 'average_studies': 2.0}


# Query 6

**Average geodesic distance to work for current Google workers.**

- Use `$geoNear` as the first stage to compute distance from a Google
  office coordinate (replace with the exact office coords).
- Join with Company collection, unwind and match company name to Google.
- Group and compute the average distance (distance stored in meters).

Replace coordinates with the correct Google office location if needed.

In [7]:
query6 = User.aggregate([
  {
    "$geoNear": {
      "near": { "type": "Point", "coordinates": [-73.989308, 40.741895] },
      "distanceField": "distance_from_google",
      "spherical": "true",
    },
  },
  {
    "$lookup": {
      "from": "Company",
      "localField": "companies",
      "foreignField": "email",
      "as": "c_details",
    },
  },
  { "$unwind": "$c_details" },
  { "$match": { "c_details.name": { "$regex": "Google", "$options": "i" } } },
  {
    "$group": {
      "_id": "google_workers",
      "average_distance_meters": { "$avg": "$distance_from_google" },
    },
  },
]);

for d in query6:
    print(d)

{'_id': 'google_workers', 'average_distance_meters': 7216157.403580193}


# Query 7

**Top 3 universities that most often appear as study centers.**

- Join users with Educational_Center on educational_centers.email.
- Unwind ed_details and group by ed_details.name counting occurrences.
- Sort by count descending and limit to the top three.

In [21]:
query7 = Educational_Center.aggregate([
  {
    "$lookup": {
      "from": "User",
      "localField": "email",                      
      "foreignField": "educational_centers.email", 
      "as": "users",
    },
  },
  { "$unwind": "$users" },
  {
    "$group": {
      "_id": "$_id",
      "educational_center": { "$first": "$$ROOT" },
      "nb_users": { "$sum": 1 }, 
    },
  },
  { "$sort": { "nb_users": -1 } }, 
  { "$limit": 3 },
  { "$project" : { "_id": 0, "educational_center.users": 0 }}
]);

for d in query7:
    print(d)

{'educational_center': {'_id': ObjectId('690398398b5904b9d03a3c72'), 'name': 'Escuela Técnica Superior de Ingeniería de Barcelona', 'address': 'Barcelona, España', 'email': 'info@etseib.es', 'address_loc': {'type': 'Point', 'coordinates': [2.177073, 41.38258]}}, 'nb_users': 8}
{'educational_center': {'_id': ObjectId('690398398b5904b9d03a3c70'), 'name': 'Universidad Complutense de Madrid', 'address': 'Madrid, España', 'email': 'info@ucm.es', 'address_loc': {'type': 'Point', 'coordinates': [-3.703507, 40.416782]}}, 'nb_users': 8}
{'educational_center': {'_id': ObjectId('690398398b5904b9d03a3c74'), 'name': 'Instituto de Formación Professional López', 'address': 'Madrid, España', 'email': 'info@ifpl.es', 'address_loc': {'type': 'Point', 'coordinates': [-3.703507, 40.416782]}}, 'nb_users': 8}
